# Block 3 — Processing petrological data with `petropandas`

*Modern Workflow in Geology, 8 October 2026 — IPSG, Charles University*

In this notebook we take microprobe (EPMA) output **straight from the instrument's Excel
export** and turn it into mineral formulas, end-member proportions, a list of clean
analyses and publication-ready charts — without a single spreadsheet formula.

| Section | Topic |
|---|---|
| 3.1 | From Excel to a DataFrame |
| 3.2 | Units: oxides → moles → cations |
| 3.3 | Crystallochemical formulas and end-members |
| 3.4 | Data cleaning: totals and stoichiometry |
| 3.5 | Charts |
| 3.6 | Exporting results |
| 3.7 | Hands-on exercise |

Everything in sections 3.1–3.6 runs on the files shipped with the workshop
(`data/epma_export.xlsx`) and on `petropandas`' own bundled example data.

In [ ]:
from pathlib import Path

from petropandas import pd, mdb, Grt, Fsp, Bt, Ms, Amp
from petropandas import ScatterPlot, TernaryPlot, ProfilePlot
from petropandas.data import minerals

DATA = Path("data")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

## 3.1 From Excel to a DataFrame

`from petropandas import pd` gives you ordinary pandas — importing petropandas just
registers its accessors (`df.oxides`, `df.moles`, `df.cations`, `df.mineral`, `df.bulk`)
and makes `pd.read_excel` use the fast *calamine* engine.

Our file `epma_export.xlsx` looks like a typical probe export: one row per spot, a
`Point` number, a free-text `Comment`, oxide wt% columns (note the instrument calls total
iron **`FeO*`**) and the analytical `Total`.

In [ ]:
df = pd.read_excel(DATA / "epma_export.xlsx", sheet_name="Spots")
df.head()

`df.oxides()` returns only the columns petropandas recognises as oxides. Column names
are parsed as chemical formulas, and common vendor variants (`FeO*`, `FeOT`, `FeO tot`,
`H2O+`, …) are mapped to canonical names automatically — `Point`, `Comment` and `Total`
are simply ignored.

In [ ]:
df.oxides().head()

The mineral name is hidden in the `Comment` (`S12_grt_03`). One line of pandas string
handling turns it into its own column that we can filter on:

In [ ]:
df["Mineral"] = df["Comment"].str.split("_").str[1]
df["Mineral"].value_counts()

## 3.2 Units: oxides → moles → cations

Every accessor converts from whatever unit the table is currently in. The unit is
remembered in `df.attrs["petro_units"]`, so conversions can be chained — and even
reversed.

In [ ]:
grt = df.oxides.select("grt", on="Mineral")   # rows whose Mineral contains "grt"
grt.oxides().head(3)

In [ ]:
grt.moles().head(3)

In [ ]:
apfu12 = grt.cations(n_oxygens=12)      # atoms per formula unit, 12-oxygen basis
print(apfu12.attrs["petro_units"])
apfu12.head(3)

Derived quantities are added with `.calc()`. Ion columns such as `Mg{2+}` can be used
directly in the expression:

In [ ]:
apfu12.cations.calc({"XMg": "Mg{2+} / (Mg{2+} + Fe{2+})",
                     "Xsite": "Fe{2+} + Mg{2+} + Ca{2+} + Mn{2+}"}).round(3).head()

## 3.3 Crystallochemical formulas and end-members

A fixed oxygen basis is only the first step. The `df.mineral` accessor takes a
**mineral definition** that knows its oxygen basis, ideal cation total, Fe³⁺ estimation
method, site assignment and end-members.

In [ ]:
print(mdb)
for m in mdb.all():
    print(repr(m))

In [ ]:
grt.mineral.apfu(Grt).round(3).head()          # APFU incl. Fe3+/Fe2+ split (Droop 1987)

In [ ]:
grt.mineral.site_allocations(Grt).round(3).head().T   # structural formula by site

In [ ]:
grt.mineral.end_members(Grt).round(1).head()

The same three calls work for every mineral — only the mineral object changes:

In [ ]:
pl = df.oxides.select("pl", on="Mineral")
pl.mineral.end_members(Fsp).round(1).head()

In [ ]:
bt = df.oxides.select("bt", on="Mineral")
bt.mineral.end_members(Bt).round(1).head()

Amphibole is the classic case where hand calculation is painful (Fe³⁺ by Schumacher's
method, 13 end-members). Here with the bundled example dataset:

In [ ]:
amp = minerals.oxides.select("Amphibole", on="Mineral")
amp.mineral.end_members(Amp).round(1).head()

## 3.4 Data cleaning: totals and stoichiometry

Not every spot is a good analysis: cracks, holes, grain boundaries and epoxy all
produce rows that *look* fine in a spreadsheet. Two independent checks help:

1. **Analytical total** — oxide sum outside the expected window (≈ 98.5–101 wt% for
   anhydrous minerals, ≈ 94–97 for micas).
2. **Stoichiometry** — does the recalculated formula have the right number of cations
   and full sites? `check_stoichiometry` scores each criterion 0–1,
   `stoichiometry_quality` condenses them into one number.

In [ ]:
qc = grt.mineral.check_stoichiometry(Grt)
qc.round(2).sort_values("analytical_total").head(6)

In [ ]:
pd.concat([grt["Comment"], grt.oxides().sum(axis=1).rename("Total"),
           grt.mineral.stoichiometry_quality(Grt)], axis=1).sort_values("stoichiometry_quality").round(3).head(6)

Note the two `_crack` spots: their totals are far too low, but their **stoichiometry is
perfect** — the whole analysis was just scaled down, so all the ratios are preserved.
Stoichiometry alone would never catch them. The opposite case is `S12_grt_incl`, where
the beam partly hit a plagioclase inclusion: its total is perfect, but the formula is
not. Always combine both checks:

In [ ]:
def clean(data, mineral, total=(98.5, 101.5), min_quality=0.97):
    # Keep analyses with a sensible oxide total AND a good formula.
    tot = data.oxides().sum(axis=1)
    q = data.mineral.stoichiometry_quality(mineral)
    ok = tot.between(*total) & (q >= min_quality)
    print(f"{mineral.name}: kept {ok.sum()} of {len(ok)}; rejected: {list(data.loc[~ok, 'Comment'])}")
    return data.oxides.select(ok)

grt_ok = clean(grt, Grt)
pl_ok = clean(pl, Fsp)
bt_ok = clean(bt, Bt, total=(94, 97.5), min_quality=0.95)

## 3.5 Charts

Three plot classes share the same pattern: create → `.add(data, label=...)` (as many
times as you like) → `.show()` or `.savefig()`. Axes are **expressions**, so you can plot
`"Prp + Sps"` or `"Mg{2+}/(Mg{2+}+Fe{2+})"` without creating new columns first.

In [ ]:
t = TernaryPlot("An", "Ab", "Or", title="Feldspar")
t.add(minerals.oxides.select(["Plagioclase", "K-feldspar"], on="Mineral").mineral.end_members(Fsp),
      label="bundled examples", alpha=0.4)
t.add(pl_ok.mineral.end_members(Fsp), label="S12 plagioclase")
t.show()

In [ ]:
s = ScatterPlot("Si{4+}", "Fe{2+} / (Fe{2+} + Mg{2+})", xlabel="Si (apfu)", ylabel="Fe/(Fe+Mg)",
                title="Micas, 11-oxygen basis")
s.add(bt_ok.mineral.apfu(Bt), label="biotite")
s.add(df.oxides.select("ms", on="Mineral").mineral.apfu(Ms), label="muscovite")
s.show()

The second sheet holds a core-to-rim-to-core garnet traverse, indexed by distance:

In [ ]:
profile = pd.read_excel(DATA / "epma_export.xlsx", sheet_name="Grt-profile", index_col="Distance_um")
p = ProfilePlot(columns=["Alm", "Prp", "Grs", "Sps"], split="auto",
                xlabel="Distance (µm)", title="Garnet zoning")
p.add(profile.mineral.end_members(Grt), lw=2)
p.savefig(OUT / "grt_profile.pdf")    # vector graphics for publication
p.show()

## 3.6 Exporting results

Results are ordinary DataFrames, so any pandas writer works. One Excel workbook with a
sheet per result is often what co-authors want:

In [ ]:
with pd.ExcelWriter(OUT / "S12_results.xlsx") as xls:
    grt_ok.mineral.apfu(Grt).round(4).to_excel(xls, sheet_name="Grt apfu")
    grt_ok.mineral.end_members(Grt).round(2).to_excel(xls, sheet_name="Grt end-members")
    pl_ok.mineral.end_members(Fsp).round(2).to_excel(xls, sheet_name="Pl end-members")
    bt_ok.mineral.end_members(Bt).round(2).to_excel(xls, sheet_name="Bt end-members")
sorted(p.name for p in OUT.iterdir())

## 3.7 Hands-on exercise

Work with the **muscovite** analyses in `epma_export.xlsx` (`Mineral == "ms"`):

1. Select the muscovite rows and recalculate them with the `Ms` mineral definition.
2. Clean them with the `clean()` function above — which total window makes sense for a
   hydrous mineral?
3. Calculate end-members and the Si content (apfu). Is there a celadonite (phengite)
   component?
4. Plot Si (apfu) against Na/(Na+K) with `ScatterPlot` and save it as SVG to `output/`.

Write your code in the empty cells below; a worked solution follows.

In [ ]:
# your code here

In [ ]:
# your code here

<details>
<summary><b>Click here to see the solution</b></summary>

```python
# 1-3
ms = df.oxides.select("ms", on="Mineral")
ms_ok = clean(ms, Ms, total=(94, 97), min_quality=0.95)
ms_ok.mineral.end_members(Ms).round(1).describe().loc[["mean", "std"]]

# 4
s = ScatterPlot("Si{4+}", "Na{+} / (Na{+} + K{+})", xlabel="Si (apfu)", ylabel="Na/(Na+K)",
                title="White mica")
s.add(ms_ok.mineral.apfu(Ms), label="S12 muscovite")
s.savefig(OUT / "ms_si_na.svg")
s.show()
```
</details>

## Recap

| Task | Code |
|---|---|
| Load instrument export | `pd.read_excel(file, sheet_name=...)` |
| Keep oxide columns | `df.oxides()` |
| Filter rows | `df.oxides.select("grt", on="Mineral")` or a boolean mask |
| APFU on fixed oxygen basis | `df.cations(n_oxygens=12)` |
| Mineral formula / sites / end-members | `df.mineral.apfu(Grt)`, `.site_allocations(Grt)`, `.end_members(Grt)` |
| Quality control | `df.mineral.check_stoichiometry(Grt)`, `.stoichiometry_quality(Grt)` + oxide total |
| Charts | `TernaryPlot`, `ScatterPlot`, `ProfilePlot` → `.add()` → `.savefig()` |

In Block 4 you will run the same pipeline on your own data with
`04b_byod_epma.ipynb`.